# Tema aleasa

### 18. Sisteme automate bazate pe procesarea semnalelor EEG și învățare automată

Actualizat pentru 2026: extins cu BCI bazată pe imaginație motorie pentru controlul unei proteze.
Obiectiv: Obiectivul acestui proiect este dezvoltarea unei interfețe creier-calculator (BCI) care
achiziționează semnale EEG, clasifică intenția utilizatorului și transmite comanda identificată
către un dispozitiv controlat. Aplicația principală propusă este controlul unei proteze de
membru superior prin imaginația motorie a mâinii stângi și a mâinii drepte, cu un model de
clasificare care generalizează la subiecți noi fără calibrare individuală. Sunt acceptate și alte
aplicații (control robotic, conducere autonomă, automatizarea locuinței).

__Setul de date 1 – setul de date de imaginație motorie WBCIC-SHU (Yang et al., 2025)__

Înregistrări EEG multi-subiect și multi-sesiune ale imaginației motorii a mâinii stângi și a
mâinii drepte, cu 53 de canale; utilizat în ediția anterioară pentru definirea protocolului
de antrenare/validare (scenarii intra-sesiune, inter-sesiune și inter-subiect).

__Placa OpenBCI Cyton (Daisy) cu 16 canale, împreună cu casca Ultracortex Mark IV__

achiziție cu BrainFlow (interfețe Python/MATLAB) sau cu OpenBCI GUI.
• Documentație: docs.openbci.com/Cyton/CytonLanding și brainflow.readthedocs.io;
dosarele proiectelor anterioare conțin exemple de înregistrări (Fatigue state, articol
despre imaginație motorie, EEGLAB).

__Metodologie:__ 

Preprocesare în MATLAB/EEGLAB sau Python (filtrare trece-bandă și notch,
re-referențiere, segmentare în epoci, eliminarea liniei de bază, re-eșantionare); analiză
temporală, spațială și spectrală pentru confirmarea calității datelor; compararea CSP + SVM,
FBCSP + SVM, EEGNet și DeepConvNet în scenarii cu dificultate crescândă (intra-sesiune,
inter-sesiune, inter-subiect); selectarea arhitecturii cu cel mai bun compromis între
performanță și cost computațional; validare pe subiecți complet izolați de antrenare; reducerea
numărului de canale pentru a corespunde căștii disponibile; interfață cu utilizatorul
(monitorizarea predicției și a acurateței) și transmiterea comenzii către dispozitiv (de exemplu,
o proteză acționată de servomotoare prin Arduino).

__Rezultate așteptate:__

• Un clasificator de imaginație motorie independent de subiect, validat pe subiecți izolați.

• Un studiu al influenței numărului de canale asupra acurateței.

• Un lanț demonstrativ complet: EEG → preprocesare → clasificator → interfață cu
utilizatorul → proteză/robot.

![../data/assets/head.png](../data/assets/head.png)

In [ ]:
%pip install -r ../requirements.txt

In [2]:
import sys

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))


import config as cfg

In [4]:
# Import necessary libraries

import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import warnings as wn
import dask.dataframe as dd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from os.path import isfile, join

from sklearn import *
from os import listdir
from pathlib import Path
from moabb.datasets import Yang2025
from moabb.paradigms import LeftRightImagery

In [3]:
# Ignore all warnings
wn.filterwarnings('ignore')

In [5]:
# Create general variables

_data_path = cfg.PROJECT_DIR
_limit_subjects = cfg.DEFAULT_LIMIT_SUBJECTS
_data_dir = cfg.DATA_DIR

In [7]:
# Read the dataset
dataset = Yang2025()
dataset.return_all_modalities = True
print(f'Available: {dataset.subject_list}')

Available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


In [8]:
# Load some pacients
subjects = dataset.subject_list[:_limit_subjects]

In [9]:
# Set the paradigm dataset
paradigm = LeftRightImagery(fmin=8, fmax=32)

In [10]:
# Load data in small segments

writer = None
all_data = []

for sub in subjects:
    X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[sub])
    sessions = metadata['session'].unique()

    n_trials, n_channels, n_timestamps = X.shape

    X_swap = np.transpose(X, (0, 2, 1))
    X_flat = X_swap.reshape(n_trials * n_timestamps, n_channels).astype(np.float32)

    columns = [f'ch_{i}' for i in range(n_channels)]
    df_sub = pd.DataFrame(X_flat, columns=columns)

    df_sub.insert(0, 'timestamp', np.tile(np.arange(n_timestamps, dtype=np.int32), n_trials))
    df_sub.insert(0, 'trial', np.repeat(np.arange(n_trials, dtype=np.int32), n_timestamps))
    df_sub.insert(0, 'subject', np.int32(sub))
    df_sub.insert(0, 'label', np.repeat(np.array(y, dtype=str), n_timestamps))

    # Tabular data 
    sub_file = _data_dir / f'data_sub_{sub:02d}.parquet'
    df_sub.to_parquet(sub_file, engine='pyarrow', compression='snappy', index=False)

    # Cubical data
    npz_file = _data_dir / f'data_sub_{sub:02d}.npz'
    np.savez_compressed(npz_file, X=X, y=y, metadata=metadata)

    file_size_mb = sub_file.stat().st_size / (1024 * 1024)
    print(f'Save subject {sub} ({len(df_sub)} rows, {file_size_mb:.2f} MB) to file')

    del X, y, metadata, X_swap, X_flat, df_sub
    gc.collect()

if writer is not None:
    writer.close()

Save subject 1 (2400600 rows, 672.35 MB) to file
Save subject 2 (2400600 rows, 675.78 MB) to file
Save subject 3 (2400600 rows, 668.19 MB) to file


In [12]:
# Set custom column names

med_columns = cfg.MED_COLUMNS
channel_names = cfg.RAW_CHANNEL_NAMES

In [13]:
# Load all of them together
parquet_files = sorted(list(_data_dir.glob("data_sub_*.parquet")))
print(f"Files: {[f.name for f in parquet_files]}")
data = dd.read_parquet(parquet_files)

Files: ['data_sub_01.parquet', 'data_sub_02.parquet', 'data_sub_03.parquet']


In [14]:
# Displat the first n rows
data.head(n=10)

,label,subject,trial,timestamp,ch_0,ch_1,ch_2,ch_3,ch_4,ch_5,...,ch_54,ch_55,ch_56,ch_57,ch_58,ch_59,ch_60,ch_61,ch_62,ch_63
0,left_hand,1,0,0,-0.002103,-0.001852,-0.002874,-0.003164,-0.001652,-0.001154,...,0.001198,0.002832,0.003152,0.002958,0.002928,-375.0,-21.107891,-115.354019,-74.953644,-375.0
1,left_hand,1,0,1,-0.002297,-0.002012,-0.002857,-0.003467,-0.001647,-0.001074,...,0.001058,0.002794,0.003080,0.002912,0.002866,-375.0,-21.108248,-115.354019,-72.998672,-375.0
2,left_hand,1,0,2,-0.002465,-0.002167,-0.002790,-0.003735,-0.001618,-0.000980,...,0.000904,0.002692,0.002942,0.002804,0.002741,-375.0,-21.107445,-115.354370,-71.007172,-375.0
3,left_hand,1,0,3,-0.002604,-0.002315,-0.002677,-0.003964,-0.001563,-0.000877,...,0.000742,0.002534,0.002746,0.002643,0.002563,-375.0,-21.108473,-115.353523,-69.126808,-375.0
4,left_hand,1,0,4,-0.002713,-0.002453,-0.002523,-0.004151,-0.001481,-0.000769,...,0.000578,0.002331,0.002504,0.002439,0.002342,-375.0,-21.108650,-115.353256,-67.201118,-375.0
5,left_hand,1,0,5,-0.002791,-0.002581,-0.002331,-0.004293,-0.001374,-0.000662,...,0.000418,0.002094,0.002227,0.002203,0.002088,-375.0,-21.108070,-115.352676,-65.306587,-375.0
6,left_hand,1,0,6,-0.002836,-0.002694,-0.002108,-0.004388,-0.001242,-0.000559,...,0.000266,0.001832,0.001928,0.001945,0.001815,-375.0,-21.107578,-115.352051,-64.069191,-375.0
7,left_hand,1,0,7,-0.002850,-0.002792,-0.001860,-0.004436,-0.001085,-0.000465,...,0.000128,0.001557,0.001618,0.001677,0.001532,-375.0,-21.106148,-115.353882,-63.527878,-375.0
8,left_hand,1,0,8,-0.002832,-0.002872,-0.001593,-0.004435,-0.000906,-0.000383,...,0.000008,0.001280,0.001309,0.001411,0.001252,-375.0,-21.106237,-115.354729,-63.690777,-375.0
9,left_hand,1,0,9,-0.002783,-0.002932,-0.001315,-0.004388,-0.000708,-0.000318,...,-0.000090,0.001012,0.001012,0.001155,0.000984,-375.0,-21.107489,-115.355133,-64.224182,-375.0


In [15]:
# Extract dimensions of input data
y_data = data['label']
X_data = data.drop(columns=['subject', 'trial', 'timestamp'])

In [16]:
# Display the first n rows
X_data = X_data.rename(columns=dict(zip(channel_names, med_columns)))
X_data.head(n=10)

,label,Fpz,Fp1,Fp2,AF3,AF4,AF7,AF8,Fz,F1,...,PO7,PO8,Oz,O1,O2,ECG,EOG1,EOG2,EOG3,EOG4
0,left_hand,-0.002103,-0.001852,-0.002874,-0.003164,-0.001652,-0.001154,-0.000303,-0.002466,-0.002464,...,0.001198,0.002832,0.003152,0.002958,0.002928,-375.0,-21.107891,-115.354019,-74.953644,-375.0
1,left_hand,-0.002297,-0.002012,-0.002857,-0.003467,-0.001647,-0.001074,-0.000666,-0.002685,-0.002669,...,0.001058,0.002794,0.003080,0.002912,0.002866,-375.0,-21.108248,-115.354019,-72.998672,-375.0
2,left_hand,-0.002465,-0.002167,-0.002790,-0.003735,-0.001618,-0.000980,-0.001025,-0.002871,-0.002838,...,0.000904,0.002692,0.002942,0.002804,0.002741,-375.0,-21.107445,-115.354370,-71.007172,-375.0
3,left_hand,-0.002604,-0.002315,-0.002677,-0.003964,-0.001563,-0.000877,-0.001377,-0.003021,-0.002970,...,0.000742,0.002534,0.002746,0.002643,0.002563,-375.0,-21.108473,-115.353523,-69.126808,-375.0
4,left_hand,-0.002713,-0.002453,-0.002523,-0.004151,-0.001481,-0.000769,-0.001720,-0.003133,-0.003067,...,0.000578,0.002331,0.002504,0.002439,0.002342,-375.0,-21.108650,-115.353256,-67.201118,-375.0
5,left_hand,-0.002791,-0.002581,-0.002331,-0.004293,-0.001374,-0.000662,-0.002052,-0.003204,-0.003130,...,0.000418,0.002094,0.002227,0.002203,0.002088,-375.0,-21.108070,-115.352676,-65.306587,-375.0
6,left_hand,-0.002836,-0.002694,-0.002108,-0.004388,-0.001242,-0.000559,-0.002370,-0.003236,-0.003158,...,0.000266,0.001832,0.001928,0.001945,0.001815,-375.0,-21.107578,-115.352051,-64.069191,-375.0
7,left_hand,-0.002850,-0.002792,-0.001860,-0.004436,-0.001085,-0.000465,-0.002671,-0.003227,-0.003154,...,0.000128,0.001557,0.001618,0.001677,0.001532,-375.0,-21.106148,-115.353882,-63.527878,-375.0
8,left_hand,-0.002832,-0.002872,-0.001593,-0.004435,-0.000906,-0.000383,-0.002954,-0.003180,-0.003121,...,0.000008,0.001280,0.001309,0.001411,0.001252,-375.0,-21.106237,-115.354729,-63.690777,-375.0
9,left_hand,-0.002783,-0.002932,-0.001315,-0.004388,-0.000708,-0.000318,-0.003214,-0.003095,-0.003060,...,-0.000090,0.001012,0.001012,0.001155,0.000984,-375.0,-21.107489,-115.355133,-64.224182,-375.0


In [17]:
# Show the shape of the data
print("Total rows:", len(X_data))

Total rows: 7201800


In [18]:
# Describe the dataframe
X_data.describe().compute().T

,count,mean,std,min,25%,50%,75%,max
Fpz,7201800.0,-5.037437e-07,0.006081,-0.074428,-0.001769,3.803341e-07,0.003922,0.092045
Fp1,7201800.0,-2.774823e-07,0.006247,-0.162688,-0.001871,1.785449e-05,0.004373,0.117812
Fp2,7201800.0,-5.665491e-07,0.006225,-0.083909,-0.001767,1.114762e-05,0.003945,0.082268
AF3,7201800.0,-3.505568e-07,0.005201,-0.097345,-0.001715,-4.661323e-06,0.003618,0.080136
AF4,7201800.0,-3.176817e-07,0.005074,-0.076537,-0.001380,-3.897707e-06,0.003178,0.080054
...,...,...,...,...,...,...,...,...
ECG,7201800.0,-2.783542e+02,133.272752,-375.000000,-375.000000,-3.750000e+02,-68.295048,262.516296
EOG1,7201800.0,-2.662906e+02,120.902700,-375.000000,-375.000000,-2.632363e+02,-233.036301,-21.057331
EOG2,7201800.0,-1.157937e+02,105.919908,-313.411469,-205.595837,-9.525928e+01,-3.717229,48.629326
EOG3,7201800.0,2.236743e+01,131.422761,-229.212814,-79.217598,4.845427e+01,152.650269,226.498108


In [19]:
# Show more infos about the dataset
X_data.info(verbose=True, memory_usage=True)

<class 'dask.dataframe.dask_expr.DataFrame'>
Index: 7201800 entries, 0 to 303447
Data columns (total 65 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   label   7201800 non-null      string
 1   Fpz     7201800 non-null      float32
 2   Fp1     7201800 non-null      float32
 3   Fp2     7201800 non-null      float32
 4   AF3     7201800 non-null      float32
 5   AF4     7201800 non-null      float32
 6   AF7     7201800 non-null      float32
 7   AF8     7201800 non-null      float32
 8   Fz      7201800 non-null      float32
 9   F1      7201800 non-null      float32
10   F2      7201800 non-null      float32
11   F3      7201800 non-null      float32
12   F4      7201800 non-null      float32
13   F5      7201800 non-null      float32
14   F6      7201800 non-null      float32
15   F7      7201800 non-null      float32
16   F8      7201800 non-null      float32
17   FCz     7201800 non-null      float32
18   FC1     7201800 non-null      float32


In [20]:
# Check the number of partitions
print("Partitions:", X_data.npartitions)

Partitions: 9


In [21]:
# Check for null values
print(X_data.isnull().sum().compute())

label    0
Fpz      0
Fp1      0
Fp2      0
AF3      0
        ..
ECG      0
EOG1     0
EOG2     0
EOG3     0
EOG4     0
Length: 65, dtype: int64


In [25]:
# Unique class for each column
object_columns = X_data.select_dtypes(include=['str']).columns
if len(object_columns) > 0:
    print(f'Unique values for each object column:')
    for col in object_columns:
        unique_values = X_data[col].unique().compute()
        print(f'Column: {col}, Unique Values: {unique_values}')

Unique values for each object column:
Column: label, Unique Values: 0     left_hand
0    right_hand
Name: label, dtype: string
